In [1]:
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [2]:
df = pd.read_csv('qna_dataset.csv')
df.head(5)

,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee
3,What is the largest planet in our solar system?,Jupiter
4,What is the boiling point of water in Celsius?,100


In [3]:
# tokenize
def tokenize(text):
    text = text.lower()
    text = text.replace('?', '')
    text = text.replace("'", '')

    return text.split()

In [4]:
tokenize(df.iloc[0,0])

['what', 'is', 'the', 'capital', 'of', 'france']

In [5]:
# vocab
vocab = {
    '<unk>': 0
}

In [6]:
def build_vocab(row):
    token_q = tokenize(row['question'])
    token_a = tokenize(row['answer'])

    tokern_merged = token_q + token_a

    for token in tokern_merged:
        if token not in vocab:
            vocab[token] = len(vocab)


In [7]:
df.apply(build_vocab, axis=1)

0     None
1     None
2     None
3     None
4     None
      ... 
85    None
86    None
87    None
88    None
89    None
Length: 90, dtype: object

In [8]:
vocab

{'<unk>': 0,
 'what': 1,
 'is': 2,
 'the': 3,
 'capital': 4,
 'of': 5,
 'france': 6,
 'paris': 7,
 'germany': 8,
 'berlin': 9,
 'who': 10,
 'wrote': 11,
 'to': 12,
 'kill': 13,
 'a': 14,
 'mockingbird': 15,
 'harper-lee': 16,
 'largest': 17,
 'planet': 18,
 'in': 19,
 'our': 20,
 'solar': 21,
 'system': 22,
 'jupiter': 23,
 'boiling': 24,
 'point': 25,
 'water': 26,
 'celsius': 27,
 '100': 28,
 'painted': 29,
 'mona': 30,
 'lisa': 31,
 'leonardo-da-vinci': 32,
 'square': 33,
 'root': 34,
 '64': 35,
 '8': 36,
 'chemical': 37,
 'symbol': 38,
 'for': 39,
 'gold': 40,
 'au': 41,
 'which': 42,
 'year': 43,
 'did': 44,
 'world': 45,
 'war': 46,
 'ii': 47,
 'end': 48,
 '1945': 49,
 'longest': 50,
 'river': 51,
 'nile': 52,
 'japan': 53,
 'tokyo': 54,
 'developed': 55,
 'theory': 56,
 'relativity': 57,
 'albert-einstein': 58,
 'freezing': 59,
 'fahrenheit': 60,
 '32': 61,
 'known': 62,
 'as': 63,
 'red': 64,
 'mars': 65,
 'author': 66,
 '1984': 67,
 'george-orwell': 68,
 'currency': 69,
 'unit

In [9]:
def text_to_indices(token):

    if token in vocab:
        return vocab[token]
    else:
        return vocab['<unk>']
     

In [10]:
def text_to_num(row):
    num_q = list(map(text_to_indices, tokenize(row['question'])))
    num_a = list(map(text_to_indices, tokenize(row['answer'])))

    return num_q, num_a
    

In [11]:
df_num = df.apply(text_to_num, axis=1)
df_num

0                           ([1, 2, 3, 4, 5, 6], [7])
1                           ([1, 2, 3, 4, 5, 8], [9])
2                    ([10, 11, 12, 13, 14, 15], [16])
3           ([1, 2, 3, 17, 18, 19, 20, 21, 22], [23])
4            ([1, 2, 3, 24, 25, 5, 26, 19, 27], [28])
                           ...                       
85                    ([10, 308, 3, 309, 310], [311])
86    ([42, 312, 2, 313, 62, 63, 3, 314, 315], [316])
87                      ([1, 2, 3, 4, 5, 109], [317])
88      ([42, 318, 2, 62, 63, 3, 319, 5, 320], [321])
89           ([42, 137, 2, 62, 39, 3, 322, 323], [6])
Length: 90, dtype: object

In [12]:
df_num = [
    (torch.tensor(inputs, dtype=torch.long), torch.tensor(target, dtype=torch.long))
    for inputs, target in df_num
]

In [13]:
df_num

[(tensor([1, 2, 3, 4, 5, 6]), tensor([7])),
 (tensor([1, 2, 3, 4, 5, 8]), tensor([9])),
 (tensor([10, 11, 12, 13, 14, 15]), tensor([16])),
 (tensor([ 1,  2,  3, 17, 18, 19, 20, 21, 22]), tensor([23])),
 (tensor([ 1,  2,  3, 24, 25,  5, 26, 19, 27]), tensor([28])),
 (tensor([10, 29,  3, 30, 31]), tensor([32])),
 (tensor([ 1,  2,  3, 33, 34,  5, 35]), tensor([36])),
 (tensor([ 1,  2,  3, 37, 38, 39, 40]), tensor([41])),
 (tensor([42, 43, 44, 45, 46, 47, 48]), tensor([49])),
 (tensor([ 1,  2,  3, 50, 51, 19,  3, 45]), tensor([52])),
 (tensor([ 1,  2,  3,  4,  5, 53]), tensor([54])),
 (tensor([10, 55,  3, 56,  5, 57]), tensor([58])),
 (tensor([ 1,  2,  3, 59, 25,  5, 26, 19, 60]), tensor([61])),
 (tensor([42, 18,  2, 62, 63,  3, 64, 18]), tensor([65])),
 (tensor([10,  2,  3, 66,  5, 67]), tensor([68])),
 (tensor([ 1,  2,  3, 69,  5,  3, 70, 71]), tensor([72])),
 (tensor([ 1,  2,  3,  4,  5, 73]), tensor([74])),
 (tensor([10, 75, 76]), tensor([77])),
 (tensor([78, 79, 80, 81, 82, 83, 84]), 

In [14]:
inputs, target = df_num[0]
print(inputs)
print(target)

tensor([1, 2, 3, 4, 5, 6])
tensor([7])


In [15]:
class CustomDataset(Dataset):
    def __init__(self, df_num):
        super().__init__()
        self.df = df_num

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        inputs, target = self.df[index]

        return inputs, target

In [16]:
dataset = CustomDataset(df_num)

In [17]:
dataset

In [18]:
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

In [19]:
for question, answer in dataloader:
  print(question, answer[0])

tensor([[10, 55,  3, 56,  5, 57]]) tensor([58])
tensor([[ 42, 125,   2,  62,  63,   3, 126, 127]]) tensor([128])
tensor([[ 1,  2,  3,  4,  5, 73]]) tensor([74])
tensor([[  1,   2,   3,   4,   5, 113]]) tensor([114])
tensor([[ 10,  75, 111]]) tensor([112])
tensor([[ 42,   2,   3, 210, 137, 168, 211, 169]]) tensor([113])
tensor([[ 78,  79, 195,  81,  19,   3, 196, 197, 198]]) tensor([199])
tensor([[10, 11, 12, 13, 14, 15]]) tensor([16])
tensor([[42, 18,  2, 62, 63,  3, 64, 18]]) tensor([65])
tensor([[  1,   2,   3,   4,   5, 206]]) tensor([207])
tensor([[ 78,  79, 129,  81,  19,   3,  21,  22]]) tensor([36])
tensor([[ 10,  75, 208]]) tensor([209])
tensor([[10, 96,  3, 97]]) tensor([98])
tensor([[ 42, 137,   2,  62,  39,   3, 322, 323]]) tensor([6])
tensor([[78, 79, 80, 81, 82, 83, 84]]) tensor([85])
tensor([[10, 29,  3, 30, 31]]) tensor([32])
tensor([[  1,   2,   3,   4,   5, 279]]) tensor([280])
tensor([[ 42, 117, 118,   3, 119,  94, 120]]) tensor([121])
tensor([[ 10,  11, 157, 158, 159

### Model

In [20]:
class RNN(nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.embedding = nn.Embedding(len(vocab), embedding_dim=50)
        self.rnn = nn.RNN(50, 64, batch_first=True)
        self.fc = nn.Linear(64, len(vocab))

    def forward(self, x):
        x = self.embedding(x)
        h, h_final = self.rnn(x)
        x = self.fc(h_final.squeeze(0))

        return x

In [21]:
x = dataset[0][0].view(1,6)
x

tensor([[1, 2, 3, 4, 5, 6]])

In [22]:
embed = nn.Embedding(len(vocab), embedding_dim=50)
embedding = embed(x)
print(embedding.shape)

rnn = nn.RNN(50, 64, batch_first=True)
h, h_f = rnn(embedding)
print(h.shape)
print(h_f.shape)

fc = nn.Linear(64, len(vocab))
logits = fc(h_f.squeeze(0))
print(logits.shape)

torch.Size([1, 6, 50])
torch.Size([1, 6, 64])
torch.Size([1, 1, 64])
torch.Size([1, 324])


In [23]:
model = RNN()
print(model)

RNN(
  (embedding): Embedding(324, 50)
  (rnn): RNN(50, 64, batch_first=True)
  (fc): Linear(in_features=64, out_features=324, bias=True)
)


In [24]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr = 0.001)

In [25]:
def train(model):

    # set training mode
    model.train()

    running_loss = 0.0

    for x, y in dataloader:

        # to avoid gradient accumulation
        optimizer.zero_grad()

        # forward pass
        logits = model(x)

        # compute loss
        loss = criterion(logits, y[0])

        # backward pass
        loss.backward()

        # update parameters
        optimizer.step()

        running_loss += loss.item()

    print(f'Loss: {running_loss/len(dataloader)}')

In [26]:
for epoch in range(25):
    print(f'Epoch {epoch}:')
    train(model)

Epoch 0:
Loss: 5.797688388824463
Epoch 1:
Loss: 5.0673638237847225
Epoch 2:
Loss: 4.2227976004282635
Epoch 3:
Loss: 3.4785252650578817
Epoch 4:
Loss: 2.8822564284006753
Epoch 5:
Loss: 2.3475460873709784
Epoch 6:
Loss: 1.8682240214612749
Epoch 7:
Loss: 1.4613004724184673
Epoch 8:
Loss: 1.119620715578397
Epoch 9:
Loss: 0.8568637847900391
Epoch 10:
Loss: 0.663846187459098
Epoch 11:
Loss: 0.5212774452235963
Epoch 12:
Loss: 0.4170870633588897
Epoch 13:
Loss: 0.3396453294489119
Epoch 14:
Loss: 0.28208739732702576
Epoch 15:
Loss: 0.23657104017006025
Epoch 16:
Loss: 0.20297363566027748
Epoch 17:
Loss: 0.17415112927556037
Epoch 18:
Loss: 0.151433242774672
Epoch 19:
Loss: 0.13113059128324192
Epoch 20:
Loss: 0.11411778935127788
Epoch 21:
Loss: 0.10097391046583652
Epoch 22:
Loss: 0.08885312390824159
Epoch 23:
Loss: 0.07905106755594413
Epoch 24:
Loss: 0.07019400188906325


In [ ]:
def predict(model, question):
    token = tokenize(question)
    num_q = torch.tensor(list(map(text_to_indices, token)), dtype=torch.long)
    num_q = num_q.unsqueeze(0)

    # evaluation mode
    model.eval()

    with torch.no_grad():
        logits = model(num_q)

        y_probs = torch.nn.functional.softmax(logits, dim=1)

        value, index = torch.max(y_probs, dim=1)

        if value > 0.5:
            return list(vocab.keys())[index]
        else:
            return "I don't know."


In [36]:
predict(model, 'Who was the one who discovered penicillin?')

'alexander-fleming'